# Chen 2020 — стационарная 3D-ванна (BOP, I = 95 A)

Воспроизведение случая bead-on-plate из Chen et al., *Appl. Math. Model.* 81 (2020) 386–400 (Table 1–2),
на уровне **ванны без плазмы**: система координат едет с горелкой, тепло/ток/давление — гауссовы граничные условия.

**Включено**
- Полупластина \(y\ge 0\); дуга неподвижна в \(x=x_\mathrm{arc}\)
- Поток твёрдого металла со скоростью \(-v_w\); Дарси тянет кашу к этой скорости
- Марангони, плавучесть, зона каши, смесь AA5754 / AA4043 по \(Y_d\)
- Усреднённая капля (масса, импульс вниз, энтальпия) по смыслу Murphy (2013)
- Лоренц в металле: осесимметричный потенциал → \(\mathbf{j}\times\mathbf{B}\) под дугой
- Усиление RH из баланса массы (параболический валик), не из VOF

**Не включено**
- Плазма Ar–Al, пар металла, деформация поверхности (minimize surface energy)

**Допущения (нет в PDF статьи):** \(U=18.5\,\mathrm{V}\), \(\eta=0.75\), \(r_q=3\,\mathrm{mm}\), \(r_j=2.5\,\mathrm{mm}\), \(v_\mathrm{drop}=0.6\,\mathrm{m/s}\), \(p_\mathrm{arc,max}=200\,\mathrm{Pa}\).

## Параметры Table 1 (BOP)

| Параметр | Значение |
|---|---|
| Ток | 95 A |
| Подача проволоки | 4.3 м/мин |
| Скорость сварки | 0.9 м/мин |
| Частота капель | 93 Гц |
| Толщина | 3 мм |
| Проволока / пластина | AA4043 / AA5754 |
| Диаметр проволоки | 1.2 мм |

Ожидание Chen Table 2 (модель): RH 1.54 мм, WD 1.76 мм, BW 5.54 мм, \(\gamma=0.59\).

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from weldpool_chen import WeldPoolChen, Material, bop_95A, quick_bop_95A

ROOT = Path('.').resolve()
RESULTS = ROOT / 'results_95A.npz'

TABLE2 = {
    'measured': {'RH_mm': 1.47, 'WD_mm': 1.85, 'BW_mm': 5.92, 'gamma': 0.62},
    'chen_paper': {'RH_mm': 1.54, 'WD_mm': 1.76, 'BW_mm': 5.54, 'gamma': 0.59},
}

QUICK = False  # True → грубая сетка / короткий прогон
RUN_IF_MISSING = True  # пересчитать, если нет results_95A.npz

In [ ]:
def run_case(quick: bool = False):
    proc = quick_bop_95A(t_end=0.4) if quick else bop_95A()
    model = WeldPoolChen(material=Material(), process=proc)
    print(f'P_arc = {proc.P_arc:.1f} W, grid {proc.nx}x{proc.ny}x{proc.nz}, t_end={proc.t_end}s')
    model.run(record_every=0.1, progress=True, record_fields=True)
    return model


def save_results(model, path: Path):
    d = model.diag
    np.savez_compressed(
        path,
        xc=model.xc, yc=model.yc, zc=model.zc,
        T=model.T, fL=model.fL, Yd=model.Yd,
        u=d.u_frames[-1], w=d.w_frames[-1], speed=d.speed_frames[-1],
        time=np.asarray(d.time),
        pool_depth=np.asarray(d.pool_depth),
        pool_width=np.asarray(d.pool_width),
        RH=np.asarray(d.RH),
        gamma=np.asarray(d.gamma),
        T_max=np.asarray(d.T_max),
        max_speed=np.asarray(d.max_speed),
        Fx=model.Fx, Fy=model.Fy, Fz=model.Fz,
        summary_json=json.dumps(model.summary()),
        table2_json=json.dumps(TABLE2),
        x_arc=model.proc.x_arc,
        Lz=model.proc.Lz,
        P_arc=model.proc.P_arc,
        I_arc=model.proc.I_arc,
    )
    print('saved', path)


if RESULTS.exists() and not QUICK:
    data = np.load(RESULTS, allow_pickle=True)
    summary = json.loads(str(data['summary_json']))
    print('loaded', RESULTS)
    print(summary)
    model = None
elif RUN_IF_MISSING or QUICK:
    model = run_case(quick=QUICK)
    summary = model.summary()
    if not QUICK:
        save_results(model, RESULTS)
        data = np.load(RESULTS, allow_pickle=True)
    else:
        data = None
        print(summary)
else:
    raise FileNotFoundError(RESULTS)

## Сравнение с Table 2

In [ ]:
if data is not None:
    summary = json.loads(str(data['summary_json']))

rows = [
    ('Measured (Chen)', TABLE2['measured']),
    ('ArcWeld paper', TABLE2['chen_paper']),
    ('This model', {
        'RH_mm': summary['RH_mm'],
        'WD_mm': summary['WD_mm'],
        'BW_mm': summary['BW_mm'],
        'gamma': summary['gamma'],
    }),
]
print(f"{'Source':20s}  {'RH':>7s}  {'WD':>7s}  {'BW':>7s}  {'gamma':>7s}")
for name, r in rows:
    print(f"{name:20s}  {r['RH_mm']:7.2f}  {r['WD_mm']:7.2f}  {r['BW_mm']:7.2f}  {r['gamma']:7.2f}")

In [ ]:
if data is not None:
    t = data['time']
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    ax[0].plot(t, 1e3 * data['pool_depth'], 'C0')
    ax[0].axhline(TABLE2['chen_paper']['WD_mm'], color='k', ls='--', lw=1, label='Chen WD')
    ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('WD [mm]'); ax[0].legend(); ax[0].set_title('Depth')
    ax[1].plot(t, 1e3 * data['pool_width'], 'C1')
    ax[1].axhline(TABLE2['chen_paper']['BW_mm'], color='k', ls='--', lw=1, label='Chen BW')
    ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('BW [mm]'); ax[1].legend(); ax[1].set_title('Width')
    ax[2].plot(t, 1e3 * data['RH'], 'C2')
    ax[2].axhline(TABLE2['chen_paper']['RH_mm'], color='k', ls='--', lw=1, label='Chen RH')
    ax[2].set_xlabel('t [s]'); ax[2].set_ylabel('RH [mm]'); ax[2].legend(); ax[2].set_title('Reinforcement')
    fig.tight_layout()
    plt.show()

## Поперечный профиль (как Fig. 5 Chen)

Координаты статьи: поперёк шва \(y\), высота относительно поверхности (усиление \(z>0\), проплавление \(z<0\)).

Пунктир — эта модель (параболический RH + контуры ликвидуса/солидуса на \(x=x_\mathrm{arc}\)). Маркеры — Table 2 (измерение и ArcWeld). Кривые Position 1–3 из Fig. 5 **не оцифрованы**, поэтому на графике их нет.

In [ ]:
from make_figs_95A import plot_fig5_profile

_src = data
if _src is None and model is not None:
    _src = {
        'xc': model.xc, 'yc': model.yc, 'zc': model.zc,
        'T': model.T, 'Yd': model.Yd,
        'x_arc': model.proc.x_arc,
    }

if _src is not None:
    plot_fig5_profile(
        _src, summary, TABLE2,
        ROOT / 'fig_chen_95A_profile.png',
        show=True,
    )
    print('saved', ROOT / 'fig_chen_95A_profile.png')

## Срезы: \(y=0\) (вдоль шва) и поперёк дуги

In [ ]:
def plot_fields(data):
    xc, yc, zc = data['xc'], data['yc'], data['zc']
    T, fL, speed = data['T'], data['fL'], data['speed']
    u, w = data['u'], data['w']
    x_arc = float(data['x_arc'])
    # y≈0 slice
    j0 = 0
    # cross-section at arc
    i_arc = int(np.argmin(np.abs(xc - x_arc)))

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))

    ax = axes[0, 0]
    pcm = ax.pcolormesh(1e3 * xc, 1e3 * zc, T[:, j0, :].T, shading='auto', cmap='inferno')
    ax.contour(1e3 * xc, 1e3 * zc, fL[:, j0, :].T, levels=[0.01, 0.99], colors=['w', 'cyan'], linewidths=1)
    # velocity quiver (subsample)
    sx, sz = max(len(xc)//20, 1), max(len(zc)//6, 1)
    Xq, Zq = np.meshgrid(xc[::sx], zc[::sz], indexing='ij')
    ax.quiver(1e3 * Xq, 1e3 * Zq, u[::sx, j0, ::sz], w[::sx, j0, ::sz], color='w', scale=2.0, width=0.003)
    ax.axvline(1e3 * x_arc, color='lime', ls=':', lw=1)
    ax.set_xlabel('x [mm]'); ax.set_ylabel('z [mm]'); ax.set_title('T, fL, V  (y≈0)')
    ax.set_aspect('equal'); fig.colorbar(pcm, ax=ax, label='T [K]', fraction=0.046)

    ax = axes[0, 1]
    pcm = ax.pcolormesh(1e3 * yc, 1e3 * zc, T[i_arc, :, :].T, shading='auto', cmap='inferno')
    ax.contour(1e3 * yc, 1e3 * zc, fL[i_arc, :, :].T, levels=[0.01, 0.99], colors=['w', 'cyan'], linewidths=1)
    ax.set_xlabel('y [mm]'); ax.set_ylabel('z [mm]'); ax.set_title(f'T, fL  (x=x_arc={1e3*x_arc:.1f} mm)')
    ax.set_aspect('equal'); fig.colorbar(pcm, ax=ax, label='T [K]', fraction=0.046)

    ax = axes[1, 0]
    pcm = ax.pcolormesh(1e3 * xc, 1e3 * yc, T[:, :, -1].T, shading='auto', cmap='inferno')
    ax.contour(1e3 * xc, 1e3 * yc, fL[:, :, -1].T, levels=[0.99], colors=['cyan'], linewidths=1.2)
    ax.plot(1e3 * x_arc, 0, 'g+', ms=12)
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]'); ax.set_title('Surface T (z=0)')
    ax.set_aspect('equal'); fig.colorbar(pcm, ax=ax, label='T [K]', fraction=0.046)

    ax = axes[1, 1]
    pcm = ax.pcolormesh(1e3 * xc, 1e3 * yc, speed[:, :, -1].T, shading='auto', cmap='viridis')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]'); ax.set_title('|u − u_solid| surface')
    ax.set_aspect('equal'); fig.colorbar(pcm, ax=ax, label='m/s', fraction=0.046)

    fig.tight_layout()
    fig.savefig(ROOT / 'fig_chen_95A_fields.png', dpi=140)
    plt.show()

if data is not None:
    plot_fields(data)
elif model is not None:
    # build a minimal dict from live model
    d = {
        'xc': model.xc, 'yc': model.yc, 'zc': model.zc,
        'T': model.T, 'fL': model.fL,
        'u': model.diag.u_frames[-1], 'w': model.diag.w_frames[-1],
        'speed': model.diag.speed_frames[-1],
        'x_arc': model.proc.x_arc,
    }
    plot_fields(d)

## Доля проволоки \(Y_d\) и сила Лоренца

In [ ]:
if data is not None:
    xc, yc, zc = data['xc'], data['yc'], data['zc']
    Yd = data['Yd']
    Fz = data['Fz']
    j0 = 0
    i_arc = int(np.argmin(np.abs(xc - float(data['x_arc']))))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    pcm = ax[0].pcolormesh(1e3 * xc, 1e3 * zc, Yd[:, j0, :].T, shading='auto', cmap='Blues', vmin=0, vmax=1)
    ax[0].set_xlabel('x [mm]'); ax[0].set_ylabel('z [mm]'); ax[0].set_title('Yd (wire alloy), y≈0')
    ax[0].set_aspect('equal'); fig.colorbar(pcm, ax=ax[0], fraction=0.046)
    pcm = ax[1].pcolormesh(1e3 * yc, 1e3 * zc, Fz[i_arc, :, :].T, shading='auto', cmap='coolwarm')
    ax[1].set_xlabel('y [mm]'); ax[1].set_ylabel('z [mm]'); ax[1].set_title('Fz Lorentz at x_arc')
    ax[1].set_aspect('equal'); fig.colorbar(pcm, ax=ax[1], fraction=0.046, label='N/m3')
    fig.tight_layout()
    plt.show()

## Замечания

- Совпадение с Table 2 **не гарантируется**: тепловой поток задан (\(\eta I U\)), а не получен из самосогласованной дуги ArcWeld.
- RH здесь — следствие расхода проволоки и измеренной BW; WD и BW — прогноз тепло/гидродинамики.
- Для быстрого прогона поставьте `QUICK = True`.